# Time Series Modelling and Forecasting

**Session roadmap:**  
Graphics → Decomposition → Features → ETS → ARIMA → Dynamic Regression → Hierarchical Forecasting

---

### Setup
Run the cell below first to install and import all required packages.

In [ ]:
# Install packages (run once)
!pip install statsmodels pandas numpy matplotlib scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('All packages imported successfully!')

---
## Part 1 — Time Series Graphics

### 1.1 What is a time series?

A time series is a sequence of observations indexed in time order. Unlike a cross-sectional dataset, the **order matters**: the dependence across time is the signal we want to model.

When you first look at a series, ask yourself:

| Feature | What to look for |
|---|---|
| **Trend** | Long-run upward or downward movement |
| **Seasonality** | Repeating pattern at a fixed calendar period |
| **Cycles** | Oscillations with variable length (e.g. business cycles) |
| **Outliers** | Isolated spikes — pandemics, strikes, data errors |
| **Changing variance** | Does the spread grow with the level? |

### 1.2 Tutorial: Classic time series datasets

We will work with three datasets throughout this notebook:

- **AirPassengers** — monthly airline passengers 1949–1960 (trend + multiplicative seasonality)
- **US Retail Sales** — simulated monthly retail index (trend + additive seasonality)
- **Sunspot activity** — annual sunspot counts (cycles, no trend)

We generate them below so the notebook is self-contained.

In [ ]:
# ── AirPassengers ────────────────────────────────────────────────────────────
air_data = [
    112,118,132,129,121,135,148,148,136,119,104,118,
    115,126,141,135,125,149,170,170,158,133,114,140,
    145,150,178,163,172,178,199,199,184,162,146,166,
    171,180,193,181,183,218,230,242,209,191,172,194,
    196,196,236,235,229,243,264,272,237,211,180,201,
    204,188,235,227,234,264,302,293,259,229,203,229,
    242,233,267,269,270,315,364,347,312,274,237,278,
    284,277,317,313,318,374,413,405,355,306,271,306,
    315,301,356,348,355,422,465,467,404,347,305,336,
    340,318,362,348,363,435,491,505,404,359,310,337,
    360,342,406,396,420,472,548,559,463,407,362,405,
    417,391,419,461,472,535,622,606,508,461,390,432
]
air_idx = pd.date_range('1949-01', periods=144, freq='MS')
air = pd.Series(air_data, index=air_idx, name='Passengers (thousands)')

# ── Simulated retail sales ────────────────────────────────────────────────────
np.random.seed(42)
n = 120
t = np.arange(n)
retail_trend = 100 + 0.5 * t
retail_season = 10 * np.sin(2 * np.pi * t / 12) + 5 * np.cos(4 * np.pi * t / 12)
retail_noise = np.random.normal(0, 3, n)
retail_idx = pd.date_range('2014-01', periods=n, freq='MS')
retail = pd.Series(retail_trend + retail_season + retail_noise,
                   index=retail_idx, name='Retail Index')

# ── Sunspot numbers (classic WDC-SILSO data, annual 1700-1900 excerpt) ────────
sunspot_raw = [
    5,11,16,23,36,58,29,20,10,8,3,0,0,2,11,27,47,63,60,
    39,28,26,22,11,21,40,78,122,103,73,47,35,11,5,16,34,
    70,81,111,101,73,40,20,16,5,11,22,40,60,80,90,100,70,
    40,30,20,10,5,10,30,60,90,120,100,80,50,30,20,10,5,
    15,40,80,130,110,85,55,30,20,10,5,15,45,85,130,115,
    90,60,35,20,10,5,15,45,80,130
]
sunspot = pd.Series(sunspot_raw,
                    index=pd.date_range('1800', periods=len(sunspot_raw), freq='YS'),
                    name='Sunspot count')

print('Datasets ready.')
print(f'  AirPassengers: {len(air)} monthly obs, 1949–1960')
print(f'  Retail:        {len(retail)} monthly obs, 2014–2023')
print(f'  Sunspots:      {len(sunspot)} annual obs')

In [ ]:
# ── Tutorial: time plots side by side ─────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9))

air.plot(ax=axes[0], color='steelblue')
axes[0].set_title('AirPassengers — trend + multiplicative seasonality')
axes[0].set_ylabel('Passengers (000s)')

retail.plot(ax=axes[1], color='darkorange')
axes[1].set_title('Simulated Retail — trend + additive seasonality')
axes[1].set_ylabel('Index')

sunspot.plot(ax=axes[2], color='seagreen')
axes[2].set_title('Sunspot numbers — cycles only')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

### 1.3 Seasonal plots and seasonal subseries plots

A **seasonal plot** overlays each year on the same horizontal axis so we can directly compare, say, January 2018 with January 2019.  
A **seasonal subseries plot** draws one small panel per season and shows the within-season mean as a horizontal line.

In [ ]:
# Tutorial: seasonal plot for AirPassengers
months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']

air_df = air.to_frame()
air_df['year']  = air_df.index.year
air_df['month'] = air_df.index.month
air_wide = air_df.pivot(index='month', columns='year', values='Passengers (thousands)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Seasonal plot
for yr in air_wide.columns:
    axes[0].plot(range(1,13), air_wide[yr], marker='o', markersize=3, label=str(yr))
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(months)
axes[0].set_title('Seasonal plot — AirPassengers')
axes[0].set_ylabel('Passengers (000s)')
axes[0].legend(fontsize=8, ncol=2, title='Year')

# Seasonal subseries plot
for m in range(1, 13):
    vals = air_wide.loc[m].dropna().values
    axes[1].plot([m]*len(vals), vals, 'o', color='steelblue', alpha=0.5, markersize=4)
    axes[1].hlines(vals.mean(), m-0.3, m+0.3, colors='red', linewidths=2)
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(months)
axes[1].set_title('Seasonal subseries plot — AirPassengers\n(red = within-month mean)')
axes[1].set_ylabel('Passengers (000s)')

plt.tight_layout()
plt.show()

### 1.4 The ACF — measuring serial dependence

The **autocorrelation function (ACF)** measures the linear relationship between $y_t$ and $y_{t-k}$ for different lags $k$.

- **Slow positive decay** → trend or non-stationarity  
- **Spikes at lags $m, 2m, 3m, \ldots$** → seasonality with period $m$  
- **Most spikes inside $\pm 1.96/\sqrt{T}$** → consistent with white noise

The **PACF** removes the effect of intermediate lags and is used for ARIMA identification (see Part 4).

In [ ]:
# Tutorial: ACF of three series
fig, axes = plt.subplots(3, 2, figsize=(14, 9))

series = [('AirPassengers', air, 'steelblue'),
          ('Retail', retail, 'darkorange'),
          ('Sunspots', sunspot, 'seagreen')]

for i, (name, s, col) in enumerate(series):
    s.plot(ax=axes[i, 0], color=col)
    axes[i, 0].set_title(f'{name} — time plot')
    plot_acf(s, ax=axes[i, 1], lags=40, color=col, title=f'{name} — ACF')

plt.tight_layout()
plt.show()

---
### 🔧 Exercise 1 — Reading time plots and ACFs

The cell below generates two mystery series `A` and `B`.

**Tasks:**
1. Plot both series as time plots.
2. Plot the ACF of each series (use `lags=40`).
3. Based on what you see, describe the structure of each series in one sentence.  
   Is there trend? Seasonality? Cycles? Is the series stationary?
4. Which series looks more like white noise? How can you tell?

In [ ]:
# Mystery series — do not look at the generating code!
np.random.seed(7)
t_ = np.arange(120)
idx_ = pd.date_range('2010-01', periods=120, freq='MS')

A = pd.Series(
    50 + 0.8 * t_ + 15 * np.sin(2 * np.pi * t_ / 12) + np.random.normal(0, 4, 120),
    index=idx_, name='Series A'
)

B = pd.Series(
    np.random.normal(0, 1, 120),
    index=idx_, name='Series B'
)

# ── Your code below ──────────────────────────────────────────────────────────
# 1. Time plots
# ...

# 2. ACF plots
# ...

---
## Part 2 — Decomposition

### 2.1 Additive vs multiplicative

We split a series into interpretable components:

$$
\text{Additive:} \quad y_t = T_t + S_t + R_t
$$
$$
\text{Multiplicative:} \quad y_t = T_t \times S_t \times R_t
$$

**Rule:** If seasonal fluctuations grow with the level of the series (as in AirPassengers), prefer multiplicative — or take logs first and use additive.  
If seasonal fluctuations stay roughly constant, use additive.

### 2.2 STL — Seasonal-Trend decomposition using LOESS

STL is flexible, handles evolving seasonality, and works for any seasonal period. The two key parameters are:
- `seasonal` — window for seasonal smoother (larger = more stable)
- `trend` — window for trend smoother (auto-selected by default)

A **robust** fit down-weights outliers.

In [ ]:
# Tutorial: STL on log-AirPassengers
log_air = np.log(air)
stl = STL(log_air, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
titles = ['Data (log)', 'Trend', 'Seasonal', 'Remainder']
components = [log_air, stl.trend, stl.seasonal, stl.resid]
colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']

for ax, comp, title, col in zip(axes, components, titles, colors):
    ax.plot(comp, color=col)
    ax.set_ylabel(title, fontsize=10)
    ax.axhline(0, color='gray', lw=0.7, ls='--')

fig.suptitle('STL Decomposition — log(AirPassengers)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Trend and seasonal strength features
var_resid  = np.var(stl.resid)
F_T = max(0, 1 - var_resid / np.var(stl.trend + stl.resid))
F_S = max(0, 1 - var_resid / np.var(stl.seasonal + stl.resid))
print(f'Trend strength  F_T = {F_T:.3f}  (1 = pure trend)')
print(f'Seasonal strength F_S = {F_S:.3f}  (1 = pure seasonality)')

### 🔧 Exercise 2 — STL decomposition and features

**Tasks:**
1. Apply STL to the `retail` series (not log-transformed this time — why is that fine here?).  
   Use `seasonal=13`.
2. Plot the four components.
3. Compute $F_T$ and $F_S$ for the retail series. Compare with the values for AirPassengers above.  
   Which series has stronger seasonality? Does that match what you saw in the time plots?
4. *(Bonus)* Change `seasonal` from 7 to 25 and re-plot. How does the seasonal component change?

In [ ]:
# Your code here
# 1. STL on retail
# stl_retail = STL(retail, seasonal=13, robust=True).fit()

# 2. Plot components
# ...

# 3. Compute F_T and F_S
# ...

---
## Part 3 — ETS Models

### 3.1 The ETS family

ETS stands for **Error, Trend, Seasonality**. Each component can be:

| Component | Options |
|---|---|
| Error | Additive (A) or Multiplicative (M) |
| Trend | None (N), Additive (A), Additive Damped (Ad) |
| Seasonality | None (N), Additive (A), Multiplicative (M) |

### 3.2 Simple Exponential Smoothing (SES) — ETS(A,N,N)

For a series with no trend and no seasonality:

$$
\hat{y}_{t+h|t} = \ell_t, \qquad \ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}
$$

- Large $\alpha$ → adapts quickly to recent changes
- Small $\alpha$ → produces smoother, slower-reacting forecasts

### 3.3 From SES to Holt–Winters

| Model | ETS code | Adds |
|---|---|---|
| SES | (A,N,N) | Level only |
| Holt's linear | (A,A,N) | Level + additive trend |
| Holt's damped | (A,Ad,N) | Trend that flattens at long horizons |
| Holt–Winters additive | (A,A,A) | Level + trend + additive seasonality |
| Holt–Winters multiplicative | (A,A,M) | Level + trend + multiplicative seasonality |

In [ ]:
# Tutorial: fit ETS(A,A,M) to AirPassengers and forecast 24 months
train = air[:'1958']
test  = air['1959':]

# Holt-Winters multiplicative seasonality
hw_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='mul',
    seasonal_periods=12
).fit(optimized=True)

fc = hw_model.forecast(len(test))
fc.index = test.index

# Plot
fig, ax = plt.subplots(figsize=(13, 4.5))
train.plot(ax=ax, label='Train', color='steelblue')
test.plot(ax=ax, label='Test (actual)', color='seagreen')
fc.plot(ax=ax, label='ETS(A,A,M) forecast', color='crimson', ls='--')
ax.set_title('Holt–Winters Multiplicative — AirPassengers')
ax.legend()
plt.show()

# Accuracy
mae  = np.mean(np.abs(test.values - fc.values))
mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100
print(f'Test MAE:  {mae:.1f}')
print(f'Test MAPE: {mape:.1f}%')

# Smoothing parameters
print(f"\nSmoothing parameters:")
print(f"  alpha (level):     {hw_model.params['smoothing_level']:.3f}")
print(f"  beta  (trend):     {hw_model.params['smoothing_trend']:.3f}")
print(f"  gamma (seasonal):  {hw_model.params['smoothing_seasonal']:.3f}")

In [ ]:
# Tutorial: Residual diagnostics for ETS model
resid = hw_model.resid

fig = plt.figure(figsize=(13, 6))
gs = GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, :])
ax1.plot(resid, color='steelblue')
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_title('Residuals over time')

ax2 = fig.add_subplot(gs[1, 0])
plot_acf(resid, ax=ax2, lags=30, color='steelblue', title='ACF of residuals')

ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(resid, bins=15, color='steelblue', edgecolor='white')
ax3.set_title('Histogram of residuals')

plt.tight_layout()
plt.show()

# Ljung-Box test
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(resid, lags=[12], return_df=True)
print('Ljung-Box test (lag=12):')
print(lb)
print('\nIf p-value > 0.05, residuals are consistent with white noise.')

### 🔧 Exercise 3 — Choosing and comparing ETS models

Use the `retail` series. Split: **train = first 96 months**, **test = last 24 months**.

**Tasks:**
1. Fit three models:
   - `ETS(A,N,N)` — SES (no trend, no seasonality)
   - `ETS(A,A,N)` — Holt's linear trend (no seasonality)
   - `ETS(A,A,A)` — Holt–Winters additive
2. Forecast 24 months ahead for each and plot all three against the test data.
3. Compute MAPE for each model on the test set. Which is best and why does it make sense?
4. *(Bonus)* Run residual diagnostics on the best model. Do the residuals look like white noise?

In [ ]:
# Your code here
train_r = retail.iloc[:96]
test_r  = retail.iloc[96:]

# 1. Fit models
# model_ses = ExponentialSmoothing(train_r, trend=None, seasonal=None).fit()
# model_hlt = ExponentialSmoothing(train_r, trend='add', seasonal=None).fit()
# model_hw  = ExponentialSmoothing(train_r, trend='add', seasonal='add', seasonal_periods=12).fit()

# 2. Forecast and plot
# ...

# 3. MAPE
# ...

---
## Part 4 — ARIMA Models

### 4.1 The ARIMA(p, d, q) model

$$
\phi(B)(1-B)^d y_t = c + \theta(B)\varepsilon_t
$$

- **AR($p$)**: the value at time $t$ depends on the previous $p$ values
- **I($d$)**: $d$ differences are taken to achieve stationarity
- **MA($q$)**: the error at time $t$ depends on the previous $q$ shocks

ARIMA modelling starts from **making the series stationary** (stable mean and variance).

### 4.2 Identification guide

After differencing to stationarity:

| ACF | PACF | Suggest |
|---|---|---|
| Cuts off after lag $q$ | Tails off | MA($q$) |
| Tails off | Cuts off after lag $p$ | AR($p$) |
| Both tail off | Both tail off | ARMA($p$,$q$) |
| Slow decay | — | More differencing needed |

### 4.3 Seasonal ARIMA: ARIMA$(p,d,q)(P,D,Q)_m$

The seasonal part adds AR/MA terms at multiples of the seasonal period $m$.

In [ ]:
# Tutorial: stationarity checks and differencing
log_air = np.log(air)

fig, axes = plt.subplots(3, 2, figsize=(14, 9))

# Original log series
log_air.plot(ax=axes[0, 0], color='steelblue', title='log(Air) — original')
plot_acf(log_air, ax=axes[0, 1], lags=40, title='ACF — original')

# First difference
d1 = log_air.diff().dropna()
d1.plot(ax=axes[1, 0], color='darkorange', title='First difference')
plot_acf(d1, ax=axes[1, 1], lags=40, title='ACF — first difference')

# Seasonal difference of the first difference
d1_12 = d1.diff(12).dropna()
d1_12.plot(ax=axes[2, 0], color='seagreen', title='Seasonal + first difference')
plot_acf(d1_12, ax=axes[2, 1], lags=40, title='ACF — seasonal + first diff')

plt.tight_layout()
plt.show()

# ADF test on doubly-differenced series
adf_stat, adf_p, *_ = adfuller(d1_12)
print(f'ADF test on doubly-differenced series: stat={adf_stat:.2f}, p={adf_p:.4f}')
print('Stationary.' if adf_p < 0.05 else 'Not stationary — consider more differencing.')

In [ ]:
# Tutorial: PACF for order selection
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(d1_12, ax=axes[0], lags=36, title='ACF — doubly differenced log(Air)')
plot_pacf(d1_12, ax=axes[1], lags=36, title='PACF — doubly differenced log(Air)')
plt.tight_layout()
plt.show()
print('Hint: seasonal spike at lag 12 in ACF → seasonal MA(1). Spike at lag 1 in ACF → MA(1).')
print('Candidate: ARIMA(0,1,1)(0,1,1)[12]')

In [ ]:
# Tutorial: fit SARIMA(0,1,1)(0,1,1)[12] — the "airline model"
train_air = np.log(air[:'1958'])
test_air  = np.log(air['1959':])

sarima = SARIMAX(
    train_air,
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    trend='n'
).fit(disp=False)

print(sarima.summary().tables[1])

# Forecast on log scale, back-transform
fc_log = sarima.forecast(steps=len(test_air))
fc_orig = np.exp(fc_log)
test_orig = air['1959':]

fig, ax = plt.subplots(figsize=(13, 4.5))
air[:'1958'].plot(ax=ax, label='Train', color='steelblue')
test_orig.plot(ax=ax, label='Test (actual)', color='seagreen')
fc_orig.plot(ax=ax, label='SARIMA(0,1,1)(0,1,1)[12]', color='crimson', ls='--')
ax.set_title('Airline Model — SARIMA(0,1,1)(0,1,1)[12]')
ax.legend()
plt.show()

mape_sarima = np.mean(np.abs((test_orig.values - fc_orig.values) / test_orig.values)) * 100
print(f'Test MAPE: {mape_sarima:.1f}%')

### 🔧 Exercise 4 — ARIMA identification and fitting

Use the `retail` series (train/test split same as Exercise 3).

**Tasks:**
1. Plot the ACF and PACF of the raw retail series. Is it stationary?
2. Apply a first difference. Plot the ACF and PACF again. Run an ADF test.  
   Does one difference make it stationary?
3. Based on the ACF/PACF, propose an ARIMA order. Fit your model with `SARIMAX`.
4. Forecast 24 months ahead and plot against the test data.
5. Compare the MAPE with the best ETS model from Exercise 3.  
   Which family performs better on this series?

**Hint:** The retail series has monthly seasonality. Try both a non-seasonal ARIMA (using Fourier terms is one option) and a SARIMA with $m=12$.

In [ ]:
# Your code here

# 1. ACF/PACF of raw retail
# ...

# 2. Difference and re-check stationarity
# retail_d1 = train_r.diff().dropna()
# ...

# 3. Fit SARIMA
# sarima_r = SARIMAX(train_r, order=(?, 1, ?), seasonal_order=(?, 1, ?, 12)).fit(disp=False)

# 4. Forecast and plot
# ...

# 5. Compare MAPE
# ...

---
## Part 5 — Dynamic Regression

### 5.1 Why ordinary regression fails for time series

Ordinary least squares assumes the errors $\eta_t$ are i.i.d. When they are autocorrelated:
- Coefficient estimates are **inefficient**
- Standard errors and p-values are **misleading**
- AIC comparisons can be **unreliable**
- **Spurious regression** becomes more likely

### 5.2 Regression with ARIMA errors

$$
y_t = \beta_0 + \beta_1 x_{1,t} + \cdots + \beta_k x_{k,t} + \eta_t
$$
$$
\phi(B)(1-B)^d \eta_t = \theta(B)\varepsilon_t
$$

Only $\varepsilon_t$ is white noise. We model the serial dependence through the ARIMA error $\eta_t$.

**Seasonal effects** can be handled with dummy variables or **Fourier terms** — the latter are especially useful when the seasonal period is long.

In [ ]:
# Tutorial: dynamic regression with Fourier terms for retail series
def fourier_terms(index, period, K):
    """Generate K pairs of sin/cos Fourier terms."""
    t = np.arange(len(index))
    cols = {}
    for k in range(1, K + 1):
        cols[f'sin_{k}'] = np.sin(2 * np.pi * k * t / period)
        cols[f'cos_{k}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)

# Build features for full series, then split
K = 3  # number of Fourier pairs
fourier = fourier_terms(retail.index, period=12, K=K)
trend_var = pd.Series(np.arange(len(retail)), index=retail.index, name='trend')
X = pd.concat([trend_var, fourier], axis=1)

X_train = X.iloc[:96]
X_test  = X.iloc[96:]

# Dynamic regression: regression + AR(2) errors
dynreg = SARIMAX(
    train_r,
    exog=X_train,
    order=(2, 0, 0),     # AR(2) errors
    trend='c'
).fit(disp=False)

print(dynreg.summary().tables[1])

In [ ]:
# Forecast and diagnose
fc_dyn = dynreg.forecast(steps=len(test_r), exog=X_test)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Forecast plot
train_r.plot(ax=axes[0], label='Train', color='steelblue')
test_r.plot(ax=axes[0], label='Test (actual)', color='seagreen')
fc_dyn.plot(ax=axes[0], label='Dynamic Reg (Fourier + AR2 errors)', color='crimson', ls='--')
axes[0].legend()
axes[0].set_title('Dynamic Regression — Retail series')

# Residual ACF
plot_acf(dynreg.resid, ax=axes[1], lags=30,
         title='ACF of innovation residuals')

plt.tight_layout()
plt.show()

mape_dyn = np.mean(np.abs((test_r.values - fc_dyn.values) / test_r.values)) * 100
print(f'Dynamic Regression MAPE: {mape_dyn:.1f}%')

### 🔧 Exercise 5 — Dynamic regression with a covariate

We simulate a scenario where sales are partly driven by an advertising spend variable.

**Tasks:**
1. Run the setup cell to create `sales` and `adspend`.
2. Fit an ordinary OLS regression (`sm.OLS`) and check the ACF of residuals.  
   Is OLS appropriate here?
3. Fit a dynamic regression (`SARIMAX`) with `adspend` as a covariate and ARMA errors.  
   Try `order=(1,0,0)` for the errors.
4. Compare the coefficient on `adspend` between OLS and dynamic regression.  
   Does accounting for autocorrelation change the estimate?
5. Check the innovation residuals. Do they look like white noise now?

In [ ]:
# Setup: sales driven by advertising, with AR(1) noise
np.random.seed(99)
n_ex5 = 100
adspend = pd.Series(
    50 + 10 * np.random.randn(n_ex5),
    index=pd.date_range('2015-01', periods=n_ex5, freq='MS'),
    name='AdSpend'
)
ar_noise = np.zeros(n_ex5)
for i in range(1, n_ex5):
    ar_noise[i] = 0.7 * ar_noise[i-1] + np.random.randn()

sales = pd.Series(
    200 + 3.0 * adspend.values + ar_noise,
    index=adspend.index,
    name='Sales'
)

fig, axes = plt.subplots(2, 1, figsize=(13, 5))
sales.plot(ax=axes[0], label='Sales', color='steelblue')
axes[0].legend()
adspend.plot(ax=axes[1], label='Ad Spend', color='darkorange')
axes[1].legend()
plt.tight_layout()
plt.show()

# Your code continues below
# 2. OLS
# X_ols = sm.add_constant(adspend)
# ols_fit = sm.OLS(sales, X_ols).fit()
# ...

# 3. Dynamic regression
# dynreg5 = SARIMAX(sales, exog=adspend, order=(1, 0, 0)).fit(disp=False)
# ...

---
## Part 6 — Hierarchical Forecasting (Bonus study and excercise)

### 6.1 The coherence problem

When time series have an aggregation structure (e.g. total → region → store), forecasts should **add up correctly** across all levels. This property is called **coherence**.

If you forecast each series independently, coherence is usually violated by chance.

### 6.2 Three classic approaches

| Approach | How | Pros | Cons |
|---|---|---|---|
| **Bottom-up** | Forecast bottom level, aggregate up | Preserves granular detail | Bottom series may be noisy |
| **Top-down** | Forecast total, disaggregate by historical proportions | Simple, stable aggregate | Loses lower-level dynamics |
| **MinT** | Reconcile all-level base forecasts jointly | Optimal in theory | Requires covariance estimation |

### 6.3 Tutorial: a simple two-level hierarchy

In [ ]:
# Tutorial: two-level hierarchy — Total split into Region A and Region B
np.random.seed(21)
n_h = 60
idx_h = pd.date_range('2019-01', periods=n_h, freq='MS')
t_h = np.arange(n_h)

# True bottom-level series
A = pd.Series(80 + 0.6*t_h + 8*np.sin(2*np.pi*t_h/12) + np.random.normal(0,3,n_h), index=idx_h, name='Region A')
B = pd.Series(40 + 0.3*t_h + 4*np.sin(2*np.pi*t_h/12 + 1) + np.random.normal(0,2,n_h), index=idx_h, name='Region B')
Total = A + B
Total.name = 'Total'

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
Total.plot(ax=axes[0], color='steelblue', title='Total')
A.plot(ax=axes[1], color='darkorange', title='Region A')
B.plot(ax=axes[2], color='seagreen', title='Region B')
plt.tight_layout()
plt.show()

In [ ]:
# Bottom-up vs Top-down forecasting
h = 12  # forecast horizon
train_n = n_h - h

def ets_forecast(series, h):
    """Fit ETS(A,A,A) and return h-step forecast."""
    model = ExponentialSmoothing(
        series, trend='add', seasonal='add', seasonal_periods=12
    ).fit(optimized=True)
    return model.forecast(h)

# ── Bottom-up: forecast A and B, then add ────────────────────────────────────
fc_A = ets_forecast(A.iloc[:train_n], h)
fc_B = ets_forecast(B.iloc[:train_n], h)
fc_BU_total = fc_A + fc_B  # coherent by construction
fc_BU_total.index = idx_h[train_n:]

# ── Top-down: forecast Total, split by historical proportions ─────────────────
prop_A = A.iloc[:train_n].mean() / Total.iloc[:train_n].mean()
prop_B = 1 - prop_A
fc_Total_only = ets_forecast(Total.iloc[:train_n], h)
fc_TD_A = prop_A * fc_Total_only
fc_TD_B = prop_B * fc_Total_only
fc_TD_A.index = fc_TD_B.index = idx_h[train_n:]

# ── Evaluate: MAPE on the Total series ───────────────────────────────────────
actual_total = Total.iloc[train_n:]
mape_bu = np.mean(np.abs((actual_total.values - fc_BU_total.values) / actual_total.values)) * 100
mape_td = np.mean(np.abs((actual_total.values - fc_Total_only.values) / actual_total.values)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, fc_total, label, col in [
    (axes[0], fc_BU_total, f'Bottom-up (MAPE={mape_bu:.1f}%)', 'crimson'),
    (axes[1], fc_Total_only, f'Top-down (MAPE={mape_td:.1f}%)', 'purple')
]:
    Total.plot(ax=ax, color='steelblue', label='Actual Total')
    fc_total.plot(ax=ax, color=col, ls='--', label=label)
    ax.axvline(idx_h[train_n], color='gray', ls=':', lw=1.5)
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Bottom-up MAPE (Total): {mape_bu:.1f}%')
print(f'Top-down  MAPE (Total): {mape_td:.1f}%')

### 🔧 Exercise 6 — Coherence and reconciliation

**Tasks:**
1. The cell above shows bottom-up and top-down forecasts for the **total**. Now check **region-level** accuracy.  
   Compute MAPE for Region A and Region B separately for both approaches.
2. Is the top-down forecast for Region A coherent (does `fc_TD_A + fc_TD_B == fc_Total_only`)?  
   Check this numerically.
3. Suppose you forecast Region A and Region B independently (not adding to the top-down total).  
   Create a "naive independent" forecast and check whether `fc_A + fc_B` matches `fc_Total_only`.  
   This illustrates the coherence problem.
4. *(Discussion)* When would you prefer bottom-up over top-down? When would you prefer top-down?  
   Write your answer as a markdown cell.

In [ ]:
# Your code here

# 1. MAPE for Region A and Region B
actual_A = A.iloc[train_n:]
actual_B = B.iloc[train_n:]

# Bottom-up region MAPE
# mape_bu_A = ...
# mape_bu_B = ...

# Top-down region MAPE
# mape_td_A = ...
# mape_td_B = ...

# 2. Check coherence of top-down
# print(np.allclose(fc_TD_A + fc_TD_B, fc_Total_only))

# 3. Naive independent forecasts coherence check
# print(np.allclose(fc_A + fc_B, fc_Total_only))

*Your discussion answer here (double-click to edit)*

**When to prefer bottom-up:**  
...

**When to prefer top-down:**  
...

---
## Part 7 — Capstone Exercise

### Putting it all together

The cell below loads a new series: **monthly electricity demand** (simulated, with trend, seasonality, and a known temperature covariate).

Work through the full forecasting workflow:

1. **Plot** the series. Describe its structure: trend? seasonality? changing variance?
2. **Decompose** with STL. Compute $F_T$ and $F_S$.
3. **Fit an ETS model**. Choose the specification based on what you saw. Forecast 12 months.
4. **Fit a SARIMA model**. Check stationarity, use ACF/PACF to guide the order. Forecast 12 months.
5. **Fit a dynamic regression** using the `temperature` covariate and ARMA errors. Forecast 12 months (use test-period temperature values as the exogenous input).
6. **Compare all three models** using MAPE on the test set. Which performs best? Why?
7. **Residual diagnostics**: run the Ljung-Box test on each model's residuals. Which models pass?
8. *(Bonus)* Combine the ETS and SARIMA forecasts as a simple average. Does the combination improve on either individual model?

In [ ]:
# Setup: electricity demand with temperature covariate
np.random.seed(2024)
n_cap = 108
idx_cap = pd.date_range('2015-01', periods=n_cap, freq='MS')
t_cap = np.arange(n_cap)

# Temperature: higher in summer (southern hemisphere: peaks in Jan/Feb)
temperature = pd.Series(
    20 + 8 * np.cos(2 * np.pi * t_cap / 12) + np.random.normal(0, 1.5, n_cap),
    index=idx_cap, name='Temperature (°C)'
)

# Demand: trend + seasonal + temperature effect + AR noise
ar_noise_cap = np.zeros(n_cap)
for i in range(1, n_cap):
    ar_noise_cap[i] = 0.4 * ar_noise_cap[i-1] + np.random.normal(0, 5)

demand = pd.Series(
    500
    + 1.2 * t_cap
    + 60 * np.cos(2 * np.pi * t_cap / 12)        # seasonal: peaks in winter (cooling demand)
    + 4.0 * temperature.values                    # temperature driver
    + ar_noise_cap,
    index=idx_cap, name='Electricity demand (GWh)'
)

# Train/test split: last 12 months as test
train_cap = demand.iloc[:-12]
test_cap  = demand.iloc[-12:]
temp_train = temperature.iloc[:-12]
temp_test  = temperature.iloc[-12:]

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
demand.plot(ax=axes[0], color='steelblue', title='Electricity Demand')
axes[0].axvline(test_cap.index[0], color='red', ls='--', lw=1.5, label='Train/test split')
axes[0].legend()
temperature.plot(ax=axes[1], color='darkorange', title='Temperature')
plt.tight_layout()
plt.show()

print('Ready. Now work through steps 1–8 in the cells below.')

In [ ]:
# Step 1: Time plot + describe
# ...

In [ ]:
# Step 2: STL decomposition and feature strengths
# ...

In [ ]:
# Step 3: ETS model
# ...

In [ ]:
# Step 4: SARIMA model
# ...

In [ ]:
# Step 5: Dynamic regression with temperature
# ...

In [ ]:
# Step 6: Compare MAPE + Step 7: Ljung-Box + Step 8 (bonus): combination
# ...

---
## Summary — Key Takeaways

| Concept | Key idea |
|---|---|
| **Time plots + ACF** | Always start here. They determine your modelling strategy. |
| **Decomposition (STL)** | Separate trend, seasonal, remainder. Compute $F_T$ and $F_S$. |
| **ETS** | Component-driven. Great for smooth trend/seasonal structure. |
| **ARIMA** | Dependence-driven. Needs stationarity. Use ACF/PACF for order. |
| **Dynamic Regression** | Combine interpretable covariates with ARIMA error dynamics. |
| **Hierarchical** | Coherence matters. Bottom-up, top-down, or reconciliation. |
| **Residual diagnostics** | Always check. A useful model leaves only noise behind. |

> **Final message:** The best model is rarely the fanciest one — it is the one whose assumptions match the data and whose residuals behave like noise.

---
### Further reading

- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed., Python version): https://otexts.com/fpppy/
- `statsmodels` documentation: https://www.statsmodels.org